In [2]:
import boto3
import json
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
client = boto3.client(
    'bedrock-runtime',
    region_name=os.getenv('AWS_REGION'),
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

# Use Amazon Nova Lite (no use-case form required)
# Switch to 'anthropic.claude-3-5-sonnet-20241022-v2:0' once you submit the Anthropic use case form
model_id = 'apac.anthropic.claude-sonnet-4-20250514-v1:0'

In [4]:
def add_user_message(messages, text):
    user_message={
        'role': 'user',
        'content': [
            {'text': text}
        ]
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message={
        'role': 'assistant',
        'content': [
            {'text': text}
        ]
    }
    messages.append(assistant_message)

def chat(messages):
    response = client.converse(
        modelId=model_id,
        messages=messages
    )
    return response['output']['message']['content'][0]['text']



In [5]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "whats 1+1?"
add_user_message(messages, "whats 1+1?")

answer = chat(messages)

# Pass the list of messages into chat to get an answer

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add in the user's followup question
add_user_message(messages, "whats 2+2?")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)

add_assistant_message(messages, answer)

In [6]:
# Make an initial list of messages
chat_messages = []

# Use a 'while True' loop to run the chatbot forever
while True:
    # Get user input
    user_input = input("> ")

    # Type 'exit' to stop the chat
    if user_input.strip().lower() == 'exit':
        print("Ending chat. Goodbye!")
        break

    print(f"> {user_input}")

    # Add user's input to list of messages
    add_user_message(chat_messages, user_input)
    # Send list of messages to the API
    text = chat(chat_messages)
    # Add generated text to list of messages
    add_assistant_message(chat_messages, text)
    # Print the generated text
    print(text)

> hello
Hello! How are you doing today? Is there anything I can help you with?
Ending chat. Goodbye!


In [7]:
# Diagnostic: list all foundation models available in your account/region
# bedrock_mgmt = boto3.client(
#     'bedrock',
#     region_name=os.getenv('AWS_REGION'),
#     aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
#     aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
# )

# models = bedrock_mgmt.list_foundation_models(byOutputModality='TEXT')
# for m in models['modelSummaries']:
#     print(f"{m['modelId']:70s}  status={m.get('modelLifecycle', {}).get('status', 'N/A')}")